# Modèle hybride métriques + CodeBERT — première version

Fusion des métriques logicielles et des représentations du code source produites par CodeBERT.

> **Exécution** : notebook conçu pour Google Colab. Les chemins pointent vers Google Drive (`/content/drive/MyDrive/bug/...`). Voir [`data/README.md`](../../data/README.md) pour obtenir les données et adapter les chemins.

*Nom d'origine dans Colab : `t_final`*


# New Section

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# New Section

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"  # Empêche transformers de charger TensorFlow

# Imports standards
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from transformers import RobertaTokenizer, RobertaModel
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 10
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    EMBEDDING_DIM = 768
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.1
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'
    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# Transformer Model
class TransBugModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_embed):
        metrics = metrics.unsqueeze(1)  # [B, 1, D]
        memory = self.encoder(metrics)
        output = self.decoder(code_embed, memory)
        return self.classifier(output[:, 0, :])

# Data preprocessing

def preprocess_metrics(df):
    df = df.drop(columns=['name'], errors='ignore')
    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    return features_scaled, df['bug'].values, scaler

def load_code_and_metrics(metrics_path, code_path):
    df_metrics = pd.read_csv(metrics_path)
    with open(code_path, 'r') as f:
        code_lines = f.readlines()
    return df_metrics, code_lines

def prepare_dataset(metrics_paths, code_paths, tokenizer, fit_smote=True):
    metrics_list, labels_list, code_list = [], [], []
    for mp, cp in zip(metrics_paths, code_paths):
        df_metrics, code_lines = load_code_and_metrics(mp, cp)
        features, labels, _ = preprocess_metrics(df_metrics)
        metrics_list.append(features)
        labels_list.append(labels)
        code_list.extend(code_lines)

    X = np.vstack(metrics_list)
    y = np.concatenate(labels_list)

    if fit_smote:
        smote = SMOTE()
        X, y = smote.fit_resample(X, y)
        code_list = code_list[:len(X)]

    code_tokens = tokenizer(code_list, padding=True, truncation=True, return_tensors='pt')
    return X, code_tokens, y

# Evaluation

def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)
            code_embed = codebert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            outputs = model(metrics, code_embed)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    g_mean = gmean([sens, spec])
    return cm, f1, sens, spec, g_mean, all_preds, all_labels

# Training loop

def train():
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    global codebert
    codebert = RobertaModel.from_pretrained("microsoft/codebert-base").to(Config.DEVICE)
    codebert.eval()

    # Replace with actual PROMISE file paths
    metrics_train_paths = ["/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv"]
    code_train_paths = ["/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/ant-1.3"]
    metrics_val_paths = ["/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv"]
    code_val_paths = ["/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/ant-1.7"]

    X_train, code_train_tokens, y_train = prepare_dataset(metrics_train_paths, code_train_paths, tokenizer)
    X_val, code_val_tokens, y_val = prepare_dataset(metrics_val_paths, code_val_paths, tokenizer, fit_smote=False)

    train_ds = BugDataset(X_train, code_train_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    model = TransBugModel().to(Config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0
    for epoch in range(Config.EPOCHS):
        model.train()
        for metrics, code_tokens, labels in train_loader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)
            code_embed = codebert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            outputs = model(metrics, code_embed)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        cm, f1, sens, spec, gmean, preds, labels = evaluate(model, val_loader)
        print(f"Epoch {epoch+1}: F1={f1:.4f}, G-Mean={gmean:.4f}, Sens={sens:.4f}, Spec={spec:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), Config.SAVE_PATH + 'best_model.pt')
            joblib.dump(cm, Config.SAVE_PATH + 'conf_matrix.pkl')
            joblib.dump(preds, Config.SAVE_PATH + 'val_preds.pkl')
            joblib.dump(labels, Config.SAVE_PATH + 'val_labels.pkl')

    print("\nBest model saved at:", Config.SAVE_PATH + 'best_model.pt')
    print("Classification Report:")
    print(classification_report(labels, preds))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title("Confusion Matrix")
    plt.show()

if __name__ == "__main__":
    train()


# New Section

In [ ]:
# Cross-project and cross-version bug prediction using Transformer encoder-decoder

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import gmean
from imblearn.over_sampling import SMOTE
from tqdm import tqdm
import json
import random

# ==== Configuration ====
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 20
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    EMBEDDING_DIM = 64
    N_HEADS = 4
    NUM_LAYERS = 3
    DROPOUT = 0.3
    NUM_CLASSES = 2
    RESULTS_DIR = "/content/drive/MyDrive/bug-results"

os.makedirs(Config.RESULTS_DIR, exist_ok=True)

# ==== Dataset Class ====
class BugDataset(Dataset):
    def __init__(self, metrics, codes, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.codes = torch.LongTensor(codes)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.metrics[idx], self.codes[idx], self.labels[idx]

# ==== Model ====
class TransformerBugPredictor(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding_dim = Config.EMBEDDING_DIM

        self.metrics_encoder = nn.Sequential(
            nn.Linear(in_features=30, out_features=self.embedding_dim),
            nn.LayerNorm(self.embedding_dim)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding_dim,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        self.code_embedding = nn.Embedding(vocab_size, self.embedding_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.embedding_dim,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=Config.NUM_LAYERS)

        self.classifier = nn.Sequential(
            nn.Linear(self.embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, Config.NUM_CLASSES),
        )

    def forward(self, metrics, code):
        # metrics: (B, F) -> (B, 1, D)
        encoded = self.metrics_encoder(metrics).unsqueeze(1)
        enc_out = self.encoder(encoded)

        code_embed = self.code_embedding(code)  # (B, L, D)
        dec_out = self.decoder(code_embed, enc_out)  # (B, L, D)

        pooled = torch.mean(dec_out, dim=1)  # Global average pooling
        return self.classifier(pooled)

# ==== Utils ====
def preprocess_metrics(df):
    df = df.copy()
    df['bug'] = (df['bug'] > 0).astype(int)
    y = df['bug'].values
    X = df.drop(['bug', 'name'], axis=1, errors='ignore')

    stds = X.std()
    X = X.loc[:, stds > 1e-5]  # Drop near-constant cols
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, y

def tokenize_code(code_str):
    return [hash(tok) % 5000 for tok in code_str.split()]  # simple hash-based tokenizer

def pad_sequences(sequences, max_len=100, pad_value=0):
    return np.array([s[:max_len] + [pad_value] * (max_len - len(s)) for s in sequences])

# ==== Evaluation & Logging ====
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for metrics, code, labels in loader:
            metrics, code = metrics.to(Config.DEVICE), code.to(Config.DEVICE)
            outputs = model(metrics, code)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    cm = confusion_matrix(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if tp + fn != 0 else 0
    specificity = tn / (tn + fp) if tn + fp != 0 else 0
    g_mean = gmean([sensitivity, specificity])
    return cm, f1, sensitivity, specificity, g_mean, all_labels, all_preds

def plot_and_save(cm, title, out_path):
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.savefig(out_path)
    plt.close()

# ==== Training Loop ====
def train_one_fold(train_df, val_df, fold_name):
    X_train, y_train = preprocess_metrics(train_df)
    X_val, y_val = preprocess_metrics(val_df)

    codes_train = pad_sequences([tokenize_code(c) for c in train_df['code'].values])
    codes_val = pad_sequences([tokenize_code(c) for c in val_df['code'].values])

    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    codes_train = pad_sequences([codes_train[i] for i in range(len(y_train))])

    train_dataset = BugDataset(X_train, codes_train, y_train)
    val_dataset = BugDataset(X_val, codes_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    model = TransformerBugPredictor(vocab_size=5000).to(Config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0
    for epoch in range(Config.EPOCHS):
        model.train()
        for metrics, code, labels in train_loader:
            metrics, code, labels = metrics.to(Config.DEVICE), code.to(Config.DEVICE), labels.to(Config.DEVICE)
            optimizer.zero_grad()
            outputs = model(metrics, code)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        cm, f1, sen, spe, gm, y_true, y_pred = evaluate(model, val_loader)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), f"{Config.RESULTS_DIR}/{fold_name}_best_model.pt")
            plot_and_save(cm, f"{fold_name} Confusion Matrix", f"{Config.RESULTS_DIR}/{fold_name}_cm.png")
            with open(f"{Config.RESULTS_DIR}/{fold_name}_metrics.txt", 'w') as f:
                f.write(json.dumps({"f1": f1, "gmean": gm, "sensitivity": sen, "specificity": spe}, indent=2))
            pd.DataFrame({"true": y_true, "pred": y_pred}).to_csv(f"{Config.RESULTS_DIR}/{fold_name}_preds.csv")

# ==== Cross-version Learning ====
def get_versions(project_name, project_files):
    versions = sorted([f for f in project_files if f.startswith(project_name)])
    return versions

def run_cross_version(project_name, project_files, all_data):
    versions = get_versions(project_name, project_files)
    for i in range(1, len(versions)):
        train_files = versions[:i]
        val_file = versions[i]
        train_df = pd.concat([all_data[f] for f in train_files], ignore_index=True)
        val_df = all_data[val_file]
        train_one_fold(train_df, val_df, fold_name=f"{project_name}_cv_{i}")

# ==== Main Run ====
def main():
    # Mount Google Drive manually before running
    data_dir = "/content/drive/MyDrive/bug/bug-data"
    all_csvs = []
    for root, dirs, files in os.walk(data_dir):
        all_csvs += [os.path.join(root, f) for f in files if f.endswith('.csv')]

    all_data = {}
    for file in tqdm(all_csvs):
        df = pd.read_csv(file)
        df['code'] = df['name'].astype(str)  # use name as pseudo-code for now (replace with real code)
        all_data[os.path.basename(file)] = df

    projects = set(f.split('-')[0] for f in all_data)

    for proj in projects:
        run_cross_version(proj, list(all_data.keys()), all_data)

if __name__ == '__main__':
    main()


In [ ]:
# Notebook complet pour Google Colab avec intégration de la lecture de code depuis dossiers et tous correctifs

!pip install -q transformers imbalanced-learn seaborn joblib

In [ ]:


import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from transformers import RobertaTokenizer, RobertaModel
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Fix pour erreur avec TF
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 10
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    EMBEDDING_DIM = 768
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.1
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'
    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# Modèle Transformer encoder-decoder

class TransBugModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()
        self.metric_projection = nn.Linear(input_metric_dim, Config.EMBEDDING_DIM)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_embed):
        metrics = self.metric_projection(metrics)  # [B, D] -> [B, 768]
        metrics = metrics.unsqueeze(1)             # [B, 1, 768]
        memory = self.encoder(metrics)
        output = self.decoder(code_embed, memory)
        return self.classifier(output[:, 0, :])

# Prétraitement des métriques

def preprocess_metrics(df):
    df = df.drop(columns=['name'], errors='ignore')
    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    return features_scaled, df['bug'].values, scaler

# Chargement des métriques et du code

def load_code_and_metrics(metrics_path, code_path):
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []
    for root, dirs, files in os.walk(code_path):
        for file in files:
            if file.endswith('.java'):
                with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                    code = f.read()
                    code_lines.append(code)
    return df_metrics, code_lines

# Préparation du dataset

def prepare_dataset(metrics_paths, code_paths, tokenizer, fit_smote=True):
    metrics_list, labels_list, code_list = [], [], []
    for mp, cp in zip(metrics_paths, code_paths):
        df_metrics, code_lines = load_code_and_metrics(mp, cp)
        features, labels, _ = preprocess_metrics(df_metrics)
        metrics_list.append(features)
        labels_list.append(labels)
        code_list.extend(code_lines)

    X = np.vstack(metrics_list)
    y = np.concatenate(labels_list)

    if fit_smote:
        smote = SMOTE()
        X, y = smote.fit_resample(X, y)
        code_list = code_list[:len(X)]

    code_tokens = tokenizer(code_list, padding=True, truncation=True, return_tensors='pt', max_length=512)
    return X, code_tokens, y

# Évaluation

def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)
            code_embed = codebert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            outputs = model(metrics, code_embed)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    g_mean = gmean([sens, spec])
    return cm, f1, sens, spec, g_mean, all_preds, all_labels

# Boucle d'entraînement principale

def train():
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    global codebert
    codebert = RobertaModel.from_pretrained("microsoft/codebert-base").to(Config.DEVICE)
    codebert.eval()

    # Adapter ces chemins à tes fichiers
    metrics_train_paths = ["/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv"]
    code_train_paths = ["/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/ant-1.3"]
    metrics_val_paths = ["/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv"]
    code_val_paths = ["/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/ant-1.7"]

    X_train, code_train_tokens, y_train = prepare_dataset(metrics_train_paths, code_train_paths, tokenizer)
    X_val, code_val_tokens, y_val = prepare_dataset(metrics_val_paths, code_val_paths, tokenizer, fit_smote=False)

    train_ds = BugDataset(X_train, code_train_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # 🔥 Correction ici :
    model = TransBugModel(input_metric_dim=X_train.shape[1]).to(Config.DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0
    for epoch in range(Config.EPOCHS):
        model.train()
        for metrics, code_tokens, labels in train_loader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)
            code_embed = codebert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            outputs = model(metrics, code_embed)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        cm, f1, sens, spec, gmean, preds, labels = evaluate(model, val_loader)
        print(f"Epoch {epoch+1}: F1={f1:.4f}, G-Mean={gmean:.4f}, Sens={sens:.4f}, Spec={spec:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), Config.SAVE_PATH + 'best_model.pt')
            joblib.dump(cm, Config.SAVE_PATH + 'conf_matrix.pkl')
            joblib.dump(preds, Config.SAVE_PATH + 'val_preds.pkl')
            joblib.dump(labels, Config.SAVE_PATH + 'val_labels.pkl')

    print("\nBest model saved at:", Config.SAVE_PATH + 'best_model.pt')
    print("Classification Report:")
    print(classification_report(labels, preds))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title("Confusion Matrix")
    plt.savefig(Config.SAVE_PATH + "conf_matrix.png")
    plt.show()


if __name__ == "__main__":
    train()


In [ ]:
# --- Importations ---
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from scipy.stats import gmean
from transformers import RobertaTokenizer, RobertaModel
from imblearn.over_sampling import SMOTE
import joblib

# --- Fix Tensorflow warning ---
os.environ['TRANSFORMERS_NO_TF'] = '1'

# --- Configuration ---
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 10
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    EMBEDDING_DIM = 768
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.1
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'
    os.makedirs(SAVE_PATH, exist_ok=True)

# --- Dataset personnalisé ---
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# --- Modèle Transformer ---
class TransBugModel(nn.Module):
    def __init__(self, input_metric_dim):
        super().__init__()
        self.metric_projection = nn.Linear(input_metric_dim, Config.EMBEDDING_DIM)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_embed):
        metrics = self.metric_projection(metrics)
        metrics = metrics.unsqueeze(1)
        memory = self.encoder(metrics)
        output = self.decoder(code_embed, memory)
        return self.classifier(output[:, 0, :])

# --- Fonctions de chargement ---
def preprocess_metrics(df):
    df = df.drop(columns=['name'], errors='ignore')
    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    return features_scaled, df['bug'].values, scaler

def load_code_and_metrics(metrics_path, code_path):
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []
    for root, dirs, files in os.walk(code_path):
        for file in files:
            if file.endswith('.java'):
                with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                    code = f.read()
                    code_lines.append(code)
    return df_metrics, code_lines

def prepare_dataset(metrics_paths, code_paths, tokenizer, fit_smote=True):
    metrics_list, labels_list, code_list = [], [], []
    for mp, cp in zip(metrics_paths, code_paths):
        df_metrics, code_lines = load_code_and_metrics(mp, cp)
        features, labels, _ = preprocess_metrics(df_metrics)
        metrics_list.append(features)
        labels_list.append(labels)
        code_list.extend(code_lines)

    X = np.vstack(metrics_list)
    y = np.concatenate(labels_list)

    if fit_smote:
        smote = SMOTE()
        X, y = smote.fit_resample(X, y)
        code_list = code_list[:len(X)]

    code_tokens = tokenizer(code_list, padding=True, truncation=True, return_tensors='pt', max_length=512)
    return X, code_tokens, y

def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)
            code_embed = codebert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            outputs = model(metrics, code_embed)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    g_mean = gmean([sens, spec])
    return cm, f1, sens, spec, g_mean, all_preds, all_labels

# --- Boucle principale ---
def train():
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    global codebert
    codebert = RobertaModel.from_pretrained("microsoft/codebert-base").to(Config.DEVICE)
    codebert.eval()

    # Liste des projets à utiliser
    project_name = "ant"
    versions = ["ant-1.3", "ant-1.4", "ant-1.5", "ant-1.6", "ant-1.7"]

    metrics_paths = [f"/content/drive/MyDrive/bug/bug-data/{project_name}/{v}.csv" for v in versions]
    code_paths = [f"/content/drive/MyDrive/bug/bug-data/source-code/source-code/{project_name}/{v}" for v in versions]

    # n-1 pour entraînement, dernier pour validation
    metrics_train_paths, code_train_paths = metrics_paths[:-1], code_paths[:-1]
    metrics_val_paths, code_val_paths = [metrics_paths[-1]], [code_paths[-1]]

    # Préparation des datasets
    X_train, code_train_tokens, y_train = prepare_dataset(metrics_train_paths, code_train_paths, tokenizer)
    X_val, code_val_tokens, y_val = prepare_dataset(metrics_val_paths, code_val_paths, tokenizer, fit_smote=False)

    train_ds = BugDataset(X_train, code_train_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    model = TransBugModel(input_metric_dim=X_train.shape[1]).to(Config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0
    history = {'train_loss': [], 'val_f1': [], 'val_gmean': []}

    for epoch in range(Config.EPOCHS):
        model.train()
        running_loss = 0
        for metrics, code_tokens, labels in train_loader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)
            code_embed = codebert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            outputs = model(metrics, code_embed)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        cm, f1, sens, spec, gmean, preds, labels = evaluate(model, val_loader)
        history['train_loss'].append(running_loss / len(train_loader))
        history['val_f1'].append(f1)
        history['val_gmean'].append(gmean)

        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} F1={f1:.4f} G-Mean={gmean:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), Config.SAVE_PATH + f'{project_name}_best_model.pt')
            joblib.dump(cm, Config.SAVE_PATH + f'{project_name}_conf_matrix.pkl')

    # Visualisations
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_f1'], label='Validation F1')
    plt.plot(history['val_gmean'], label='Validation G-Mean')
    plt.title('Training Progress')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    plt.grid()
    plt.savefig(Config.SAVE_PATH + f"{project_name}_training_curves.png")
    plt.show()

    print("\nBest model saved at:", Config.SAVE_PATH + f'{project_name}_best_model.pt')

# --- Démarrer l'entraînement ---
if __name__ == "__main__":
    train()
